
### Safety Surveillance - PLE Study

##### Research Question - Does exposure to ACE inhibitor increase the risk of experiencing Angioedema within 1 month after exposure start?

##### Step 1 - Import the necessary libraries


In [ ]:
%%jupyter
# import libraries
library(rD2E)
library(Strategus)
library(dplyr)

##### Step 2 - Cohort definition loading

In [ ]:
%%jupyter
tarCohortId <- XX
outCohortId <- XX
cohorts_set <- c(tarCohortId, outCohortId)
cohortDefinitionSet <- rD2E::get_cohort_definition_set(cohorts_set)


##### Step 3 - Define the network study components

In [ ]:
%%jupyter
# If you are not restricting your study to a specific time window,
# please make these strings empty
studyStartDate <- '19001201' # YYYYMMDD
studyEndDate <- '20231231'   # YYYYMMDD


# Target only (self-controlled design possible, but here we set up one-arm analysis)
targetCohortId <- tarCohortId  # ACE Inhibitor users
outcomeCohortId <- outCohortId  # Angioedema

# Time-at-risk: 1 month after exposure start
timeAtRisks <- tibble(
  label = c("One-month risk window"),
  riskWindowStart = c(0),
  startAnchor = c("cohort start"),
  riskWindowEnd = c(30),
  endAnchor = c("cohort start")
)

# Define the outcome
outcomeList <- list(
  CohortMethod::createOutcome(
    outcomeId = outcomeCohortId,
    outcomeOfInterest = TRUE
  )
)

# Define pseudo comparator (optional) — can still use T-C-O structure with single arm
targetComparatorOutcomesList <- list(
  CohortMethod::createTargetComparatorOutcomes(
    targetId = targetCohortId,
    outcomes = outcomeList
  )
)

Create settings for modules involved in the study

In [ ]:
%%Jupyter
# Setup cohort method module
cmModuleSettingsCreator <- CohortMethodModule$new()

cmAnalysisList <- list(
  CohortMethod::createCmAnalysis(
    analysisId = 1,
    description = "1-month risk of angioedema after ACE inhibitor",
    getDbCohortMethodDataArgs = CohortMethod::createGetDbCohortMethodDataArgs(
      studyStartDate = studyStartDate,
      studyEndDate = studyEndDate
    ),
    createStudyPopArgs = CohortMethod::createCreateStudyPopulationArgs(
      firstExposureOnly = TRUE,
      removeDuplicateSubjects = "keep first",
      removeSubjectsWithPriorOutcome = TRUE,
      priorOutcomeLookback = 0,
      requireTimeAtRisk = FALSE,
      riskWindowStart = timeAtRisks$riskWindowStart,
      startAnchor = timeAtRisks$startAnchor,
      riskWindowEnd = timeAtRisks$riskWindowEnd,
      endAnchor = timeAtRisks$endAnchor
    )
  )
)

cohortMethodModuleSpecifications <- cmModuleSettingsCreator$createModuleSpecifications(
  cmAnalysisList = cmAnalysisList,
  targetComparatorOutcomesList = targetComparatorOutcomesList
)


In [ ]:
%%jupyter
# Cohort Generator
cgModuleSettingsCreator <- CohortGeneratorModule$new()
cohortDefinitionShared <- cgModuleSettingsCreator$createCohortSharedResourceSpecifications(cohortDefinitionSet)
cohortGeneratorModuleSpecifications <- cgModuleSettingsCreator$createModuleSpecifications()

Create the analysis specification object - including all the modules and configurations created above

In [ ]:
%%jupyter
# Final Analysis Spec
analysisSpecifications <- createEmptyAnalysisSpecificiations() |>
  addSharedResources(cohortDefinitionShared) |>
  addModuleSpecifications(cohortGeneratorModuleSpecifications) |>
  addModuleSpecifications(cohortMethodModuleSpecifications)


##### Step 4 - Execute the Strategus study

In [ ]:
%%jupyter
study_name <- "treatment_safety_study"  # Unique study name
options <- create_options(upload_results=TRUE, study_id = study_name) # set a study_id with a unique id
options$studyId <- study_name
rD2E::run_strategus_flow(analysisSpecification = analysisSpecifications, options = options)